CTP 2 - Exercise 1 STARTER: profile customers.csv with pandas.  
Run from the 02_CTP2 folder: `python profile_customers_starter.py`  
Goal: measure at least 8 rules (2 per dimension) and copy the numbers into quality_scorecard_template.csv 

In [6]:
import pandas as pd

df = pd.read_csv("../shopfast-kit/data/customers.csv", dtype=str)   # dtype=str: keep the raw text (e.g. 01300)

In [7]:
n = len(df)
print("Rows:", n)
print(df.head())

# --- Completeness: how many empty values per column? ---
print("\nEmpty values per column:\n", df.isna().sum())

# --- Uniqueness: is customer_id unique? ---
print("\nDuplicated customer_id rows:", df["customer_id"].duplicated(keep=False).sum())

Rows: 1241
  customer_id first_name last_name                         email  \
0       10328      Alain    Boutin        alain.boutin@gmail.com   
1       10524    Thérèse    Hubert    therese.hubert@outlook.com   
2       10125     Célina   Lemaire         celina.lemaire@gmx.de   
3       10812    Aurelia    Cuevas       aurelia.cuevas@yahoo.fr   
4       10335   Cristina    Campoy  cristina.campoy@@outlook.com   

               phone  birth_date                               address  \
0  +33 6 28 32 68 42  1956-08-31                    29, rue de Rolland   
1     06.32.83.38.98  1999-01-23                      875, rue Thierry   
2     06.65.40.51.53  09-04-1986           58, avenue Charlotte Berger   
3       +34666395517  1987-01-23            Cuesta de Jacinto Querol 6   
4       +34634110809  2006-01-10  Cuesta de Constanza Pla 98 Puerta 9    

                 city postal_code country signup_date marketing_consent  \
0             Jacquot       63965      FR  2023-08-29       

TODO 1 (Uniqueness): normalise email (strip + lower case) and count people registered twice  


In [8]:
email_norm = df["email"].str.strip().str.lower()

nb_doublons = email_norm.duplicated(keep=False).sum()

print("Duplicated email:", nb_doublons)

Duplicated email: 168


TODO 2 (Validity): count emails that do not match the regex  `^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$`   (hint: .str.match)  


In [9]:
email_pattern = r"^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$"
email_valide = email_norm.str.match(email_pattern, na=False)
nb_emails_invalides = (~email_valide).sum()

print("Invalid emails:", nb_emails_invalides)

Invalid emails: 67


TODO 3 (Consistency): list the distinct values of country with value_counts() - how many spellings for 4 countries?  


In [10]:
country_counts = df["country"].value_counts()
print("\nCountry value counts:\n", country_counts)

print("\nNumber of spellings for 4 countries:", df["country"].nunique())


Country value counts:
 country
FR           530
DE           145
BE           127
ES           103
France        68
fr            53
france        41
FRA           31
Spain         14
DEU           14
Allemagne     13
Germany       12
Espagne       11
Belgium       11
BEL           11
es            11
Belgique       9
Name: count, dtype: int64

Number of spellings for 4 countries: 17


TODO 4 (Consistency): count signup_date values that are NOT in the format YYYY-MM-DD  


In [11]:
invalid_signup_dates = df["signup_date"].str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
nb_invalid_signup_dates = (~invalid_signup_dates).sum()
print(f"\nNumber of invalid signup dates: {nb_invalid_signup_dates}")


Number of invalid signup dates: 303


TODO 5 (Validity): loyalty_tier values outside bronze / silver / gold / platinum  


In [27]:
tier=["gold", "silver", "bronze", "platinum"]
invalid_tiers = df[~df["loyalty_tier"].isin(tier)]
invalid_tiers_count =len(invalid_tiers)
print("\nLoyalty tier values outside bronze / silver / gold / platinum:\n", invalid_tiers_count)




Loyalty tier values outside bronze / silver / gold / platinum:
 13


TODO 6 (Accuracy): birth dates in the future or before 1906 (convert the ISO ones with pd.to_datetime)  


In [13]:
age = pd.to_datetime(df["birth_date"], errors="coerce")
future_birth_dates = (age > pd.Timestamp.now()).sum()
old_birth_dates = (age < pd.Timestamp("1906-01-01")).sum()

print(f"\nFuture birth dates: {future_birth_dates}")
print(f"Old birth dates: {old_birth_dates}")


Future birth dates: 5
Old birth dates: 3


TODO 7 (Validity): French postal codes that do not have 5 digits  


In [19]:
is_valid_postal_code = df["postal_code"].str.match(r"^\d{5}$", na=False)
invalid_postal_codes = df[~is_valid_postal_code]
count_invalid_postal_codes = len(invalid_postal_codes)
print(f"\nInvalid count of postal codes: {count_invalid_postal_codes}")



Invalid count of postal codes: 48


Bonus: compute score_pct = 100 * (1 - failing_rows / n) for each rule  

In [50]:
score_pct_email_uniqueness = 100*(1 - nb_doublons / n)
print(f"\nScore for email uniqueness: {score_pct_email_uniqueness:.2f}%")
score_pct_validity = 100*(1 - (count_invalid_postal_codes + nb_emails_invalides + invalid_tiers_count) / n)
print(f"Score for validity: {score_pct_validity:.2f}%")
score_pct_completeness = 100*(1 - df.isna().sum().sum() / (n * len(df.columns)))
print(f"Score for completeness: {score_pct_completeness:.2f}%")
score_pct_accuracy = 100*(1 - (future_birth_dates + old_birth_dates) / n)
print(f"Score for accuracy: {score_pct_accuracy:.2f}%")
score_pct_consistency = 100*(1 - nb_invalid_signup_dates / n)
print(f"Score for consistency: {score_pct_consistency:.2f}%")




Score for email uniqueness: 86.46%
Score for validity: 89.69%
Score for completeness: 99.29%
Score for accuracy: 99.36%
Score for consistency: 75.58%
